## Example 1: dask_geopandas functionality test 1

In [ ]:
# Example 1: Basic dask-geopandas usage and geometry operations
import geopandas as gpd
import dask_geopandas as dgpd
from shapely.geometry import Point, Polygon
import pandas as pd
import numpy as np

print("Example 1: Basic Dask-GeoPandas Operations\n")

# Create sample point data
print("Creating sample point geometries:")
np.random.seed(42)
n_points = 10000
lons = np.random.uniform(-180, 180, n_points)
lats = np.random.uniform(-90, 90, n_points)
points = [Point(lon, lat) for lon, lat in zip(lons, lats)]

# Create GeoDataFrame
gdf = gpd.GeoDataFrame({
    'id': range(n_points),
    'value': np.random.uniform(0, 100, n_points),
    'geometry': points
})
print(f"  Created GeoDataFrame with {len(gdf)} points")
print(f"  Columns: {list(gdf.columns)}\n")

# Convert to Dask GeoDataFrame
print("Converting to Dask GeoDataFrame:")
ddf = dgpd.from_geopandas(gdf, npartitions=4)
print(f"  Type: {type(ddf)}")
print(f"  Partitions: {ddf.npartitions}")
print(f"  Spatial partitions: {ddf.spatial_partitions is not None}\n")

# Perform parallel geometry operations
print("Computing geometry attributes in parallel:")
x_coords = ddf.geometry.x.compute()
y_coords = ddf.geometry.y.compute()
print(f"  X coordinate range: [{x_coords.min():.2f}, {x_coords.max():.2f}]")
print(f"  Y coordinate range: [{y_coords.min():.2f}, {y_coords.max():.2f}]")

# Show sample
print("\nFirst 5 rows:")
print(ddf.head())

print("\nDask-GeoPandas enables parallel geospatial operations")

## Example 2: dask_geopandas functionality test 2

In [ ]:
# Example 2: Spatial filtering and buffer operations
import geopandas as gpd
import dask_geopandas as dgpd
from shapely.geometry import Point, Polygon
import numpy as np

print("Example 2: Spatial Filtering and Buffer Operations\n")

# Create point dataset
np.random.seed(123)
n = 8000
points = [Point(np.random.uniform(-10, 10), np.random.uniform(-10, 10)) for _ in range(n)]
gdf = gpd.GeoDataFrame({
    'id': range(n),
    'category': np.random.choice(['A', 'B', 'C'], n),
    'geometry': points
})
ddf = dgpd.from_geopandas(gdf, npartitions=8)
print(f"Created Dask GeoDataFrame: {len(gdf)} points, {ddf.npartitions} partitions\n")

# Create a region of interest polygon
print("Creating region of interest (ROI):")
roi = Polygon([(-5, -5), (5, -5), (5, 5), (-5, 5)])
print(f"  ROI: 10x10 square centered at origin\n")

# Spatial filtering - points within ROI
print("Filtering points within ROI:")
within_roi = ddf[ddf.within(roi)]
count_within = len(within_roi)
print(f"  Points within ROI: {count_within}")
print(f"  Percentage: {count_within / n * 100:.1f}%\n")

# Create buffers around points
print("Creating buffers around points (parallel):")
buffered = ddf.geometry.buffer(0.5)
buffered_areas = buffered.area.compute()
print(f"  Buffer radius: 0.5")
print(f"  Mean buffer area: {buffered_areas.mean():.4f}")
print(f"  Total buffer area: {buffered_areas.sum():.2f}")

# Filter by category and compute statistics
print("\nStatistics by category:")
for cat in ['A', 'B', 'C']:
    cat_count = len(ddf[ddf.category == cat])
    print(f"  Category {cat}: {cat_count} points")

print("\nDask-GeoPandas performs spatial filtering in parallel")

## Example 3: dask_geopandas functionality test 3

In [ ]:
# Example 3: Distance calculations and spatial relationships
import geopandas as gpd
import dask_geopandas as dgpd
from shapely.geometry import Point
import numpy as np

print("Example 3: Distance Calculations and Spatial Relationships\n")

# Create two sets of points
np.random.seed(456)
n_cities = 5000
city_points = [Point(np.random.uniform(-100, 100), np.random.uniform(-50, 50)) for _ in range(n_cities)]
gdf_cities = gpd.GeoDataFrame({
    'city_id': range(n_cities),
    'population': np.random.randint(1000, 100000, n_cities),
    'geometry': city_points
})
ddf_cities = dgpd.from_geopandas(gdf_cities, npartitions=5)
print(f"Cities dataset: {len(gdf_cities)} points, {ddf_cities.npartitions} partitions\n")

# Create a reference point (e.g., capital)
capital = Point(0, 0)
print("Reference point (capital): (0, 0)\n")

# Calculate distances to capital
print("Computing distances to capital (parallel):")
distances = ddf_cities.geometry.distance(capital).compute()
print(f"  Min distance: {distances.min():.2f}")
print(f"  Max distance: {distances.max():.2f}")
print(f"  Mean distance: {distances.mean():.2f}")
print(f"  Median distance: {distances.median():.2f}\n")

# Find cities within certain distance
print("Finding cities within 50 units of capital:")
nearby_cities = ddf_cities[ddf_cities.geometry.distance(capital) < 50]
nearby_count = len(nearby_cities)
print(f"  Cities within 50 units: {nearby_count}")
print(f"  Percentage: {nearby_count / n_cities * 100:.1f}%\n")

# Compute total population in nearby cities
print("Population statistics for nearby cities:")
nearby_pop = nearby_cities.population.sum().compute()
total_pop = ddf_cities.population.sum().compute()
print(f"  Nearby population: {nearby_pop:,}")
print(f"  Total population: {total_pop:,}")
print(f"  Percentage in nearby cities: {nearby_pop / total_pop * 100:.1f}%")

print("\nDask-GeoPandas computes spatial distances efficiently")

## Example 4: dask_geopandas functionality test 4

In [ ]:
# Example 4: Polygon operations and area calculations
import geopandas as gpd
import dask_geopandas as dgpd
from shapely.geometry import Polygon, Point
import numpy as np

print("Example 4: Polygon Operations and Area Calculations\n")

# Create random polygons (representing parcels/zones)
np.random.seed(789)
n_polygons = 3000

def create_random_polygon():
    """Create a random polygon"""
    center_x = np.random.uniform(-50, 50)
    center_y = np.random.uniform(-50, 50)
    size = np.random.uniform(1, 5)
    return Polygon([
        (center_x - size, center_y - size),
        (center_x + size, center_y - size),
        (center_x + size, center_y + size),
        (center_x - size, center_y + size)
    ])

polygons = [create_random_polygon() for _ in range(n_polygons)]
gdf_parcels = gpd.GeoDataFrame({
    'parcel_id': range(n_polygons),
    'zone_type': np.random.choice(['Residential', 'Commercial', 'Industrial'], n_polygons),
    'geometry': polygons
})
ddf_parcels = dgpd.from_geopandas(gdf_parcels, npartitions=6)
print(f"Parcels dataset: {len(gdf_parcels)} polygons, {ddf_parcels.npartitions} partitions\n")

# Calculate areas in parallel
print("Computing polygon areas (parallel):")
areas = ddf_parcels.geometry.area.compute()
print(f"  Min area: {areas.min():.2f}")
print(f"  Max area: {areas.max():.2f}")
print(f"  Mean area: {areas.mean():.2f}")
print(f"  Total area: {areas.sum():.2f}\n")

# Calculate perimeters
print("Computing polygon perimeters (parallel):")
perimeters = ddf_parcels.geometry.length.compute()
print(f"  Mean perimeter: {perimeters.mean():.2f}")
print(f"  Total perimeter: {perimeters.sum():.2f}\n")

# Statistics by zone type
print("Area statistics by zone type:")
for zone in ['Residential', 'Commercial', 'Industrial']:
    zone_parcels = ddf_parcels[ddf_parcels.zone_type == zone]
    zone_area = zone_parcels.geometry.area.sum().compute()
    zone_count = len(zone_parcels)
    print(f"  {zone}: {zone_count} parcels, total area {zone_area:.2f}")

print("\nDask-GeoPandas handles polygon operations at scale")

## Example 5: dask_geopandas functionality test 5

In [ ]:
# Example 5: Spatial joins and aggregations
import geopandas as gpd
import dask_geopandas as dgpd
from shapely.geometry import Point, Polygon
import numpy as np

print("Example 5: Spatial Joins and Aggregations\n")

# Create grid of zones (larger polygons)
print("Creating zone grid:")
zones = []
zone_names = []
for i in range(5):
    for j in range(5):
        x_min, y_min = i * 10, j * 10
        x_max, y_max = (i + 1) * 10, (j + 1) * 10
        zones.append(Polygon([(x_min, y_min), (x_max, y_min), (x_max, y_max), (x_min, y_max)]))
        zone_names.append(f"Zone_{i}_{j}")

gdf_zones = gpd.GeoDataFrame({
    'zone_name': zone_names,
    'geometry': zones
})
print(f"  Created {len(gdf_zones)} zones\n")

# Create points dataset (e.g., events/incidents)
print("Creating events dataset:")
np.random.seed(999)
n_events = 10000
event_points = [Point(np.random.uniform(0, 50), np.random.uniform(0, 50)) for _ in range(n_events)]
gdf_events = gpd.GeoDataFrame({
    'event_id': range(n_events),
    'severity': np.random.choice(['Low', 'Medium', 'High'], n_events),
    'value': np.random.uniform(1, 100, n_events),
    'geometry': event_points
})
ddf_events = dgpd.from_geopandas(gdf_events, npartitions=10)
print(f"  Created {len(gdf_events)} events, {ddf_events.npartitions} partitions\n")

# Perform spatial join
print("Performing spatial join (events to zones):")
joined = dgpd.sjoin(ddf_events, gdf_zones, how='inner', predicate='within')
print(f"  Joined records: {len(joined)}")
print(f"  Columns: {list(joined.columns)}\n")

# Aggregate events by zone
print("Aggregating events by zone:")
zone_stats = joined.groupby('zone_name').agg({
    'event_id': 'count',
    'value': ['sum', 'mean']
}).compute()
print("\nTop 5 zones by event count:")
print(zone_stats.sort_values(('event_id', 'count'), ascending=False).head())

# Count events by severity
print("\n\nEvent severity distribution:")
severity_counts = joined.groupby('severity').size().compute()
for severity, count in severity_counts.items():
    print(f"  {severity}: {count}")

print("\nDask-GeoPandas enables parallel spatial joins and aggregations")